# Módulo 10 · Práctica: diferenciación automática

Construirás tu propio motor de autograd escalar, lo comprobarás contra PyTorch y portarás el MLP de caracteres del módulo 9 a PyTorch. Es lo que ocurre dentro de `loss.backward()` cuando se entrena un LLM, solo que con tensores en lugar de escalares.

| Parte | Qué haces |
| --- | --- |
| 0 | Preparación |
| 1 | Modo forward con números duales |
| 2 | La clase `Valor` y sus reglas locales |
| 3 | `backward` con orden topológico |
| 4 | Verificación contra PyTorch |
| 5 | Una red neuronal hecha con `Valor` |
| 6 | Forward frente a reverse: medir el coste |
| 7 | PyTorch en cuatro ideas |
| 8 | El MLP de caracteres en PyTorch |
| 9 | Retos |

Completa cada `# TU CÓDIGO` y ejecuta la `# Comprobación` siguiente. Ejecución total: < 1 min en CPU.

## Parte 0 · Preparación

In [ ]:
import math, random, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)   # con tensores pequeños, un hilo es más rápido
print("PyTorch", torch.__version__)

## Parte 1 · Modo forward: números duales

Un dual es $a + b\,\varepsilon$ con $\varepsilon^2 = 0$. Evaluar $f$ en $x + \varepsilon$ da $f(x) + f'(x)\,\varepsilon$: la derivada viaja junto al valor.

$$(a+b\varepsilon)(c+d\varepsilon) = ac + (ad+bc)\,\varepsilon, \qquad g(a+b\varepsilon) = g(a) + g'(a)\,b\,\varepsilon$$

**Ejercicio 1.** Completa `__mul__` y `tanh`. Recuerda: $\tanh'(x) = 1-\tanh^2 x$.

In [ ]:
class Dual:
    def __init__(self, a, b=0.0):
        self.a, self.b = float(a), float(b)   # parte real e infinitesimal
    def __add__(self, o):
        o = o if isinstance(o, Dual) else Dual(o)
        return Dual(self.a + o.a, self.b + o.b)
    __radd__ = __add__
    def __mul__(self, o):
        o = o if isinstance(o, Dual) else Dual(o)
        return None  # TU CÓDIGO
    __rmul__ = __mul__
    def tanh(self):
        return None  # TU CÓDIGO
    def __repr__(self):
        return f"{self.a:.4f} + {self.b:.4f}ε"

In [ ]:
# Comprobación
f = lambda x: x * x * x + 2 * x            # f'(x) = 3x² + 2
r = f(Dual(2, 1))
assert r is not None and (r.a, r.b) == (12, 14), "Debe salir f(2) = 12 y f'(2) = 14"
r = (Dual(0.5, 1) * 3).tanh()
assert abs(r.b - 3 * (1 - math.tanh(1.5) ** 2)) < 1e-12, "tanh: multiplica la derivada por self.b"
print("Correcto:", f(Dual(2, 1)))

Con $n$ entradas, el modo forward necesita **una pasada por entrada** para el gradiente completo. Lo mediremos en la parte 6.

## Parte 2 · La clase `Valor`

Cada `Valor` guarda `dato`, `grad`, sus hijos `_prev` y una función `_backward` que reparte el gradiente de la salida entre sus entradas. `__add__` y `log` sirven de modelo.

**Ejercicio 2.** Completa los `_backward` de `__mul__`, `__pow__`, `tanh` y `exp`. Usa siempre `+=`: si un nodo se usa varias veces, sus contribuciones se suman.

In [ ]:
class Valor:
    def __init__(self, dato, _hijos=(), _op=''):
        self.dato, self.grad = float(dato), 0.0
        self._backward = lambda: None
        self._prev, self._op = set(_hijos), _op

    def __repr__(self):
        return f"Valor(dato={self.dato:.4f}, grad={self.grad:.4f})"

    def __add__(self, otro):
        otro = otro if isinstance(otro, Valor) else Valor(otro)
        sal = Valor(self.dato + otro.dato, (self, otro), '+')
        def _backward():
            self.grad += sal.grad
            otro.grad += sal.grad
        sal._backward = _backward
        return sal

    def __mul__(self, otro):
        otro = otro if isinstance(otro, Valor) else Valor(otro)
        sal = Valor(self.dato * otro.dato, (self, otro), '*')
        def _backward():
            pass  # TU CÓDIGO (mul)
        sal._backward = _backward
        return sal

    def __pow__(self, k):                      # k es un número
        sal = Valor(self.dato ** k, (self,), f'**{k}')
        def _backward():
            pass  # TU CÓDIGO (pow)
        sal._backward = _backward
        return sal

    def tanh(self):
        t = math.tanh(self.dato)
        sal = Valor(t, (self,), 'tanh')
        def _backward():
            pass  # TU CÓDIGO (tanh)
        sal._backward = _backward
        return sal

    def exp(self):
        e = math.exp(self.dato)
        sal = Valor(e, (self,), 'exp')
        def _backward():
            pass  # TU CÓDIGO (exp)
        sal._backward = _backward
        return sal

    def log(self):
        sal = Valor(math.log(self.dato), (self,), 'log')
        def _backward():
            self.grad += sal.grad / self.dato
        sal._backward = _backward
        return sal

    # Operaciones compuestas: su gradiente sale solo de las anteriores
    def __neg__(self): return self * -1
    def __sub__(self, otro): return self + (-otro)
    def __rsub__(self, otro): return otro + (-self)
    def __radd__(self, otro): return self + otro
    def __rmul__(self, otro): return self * otro
    def __truediv__(self, otro): return self * otro ** -1
    def __rtruediv__(self, otro): return otro * self ** -1

In [ ]:
# Comprobación: aplicamos cada regla local a mano, sin backward
a, b = Valor(3.0), Valor(-2.0)
c = a * b; c.grad = 1.0; c._backward()
assert (a.grad, b.grad) == (-2.0, 3.0), "∂(ab)/∂a = b y ∂(ab)/∂b = a"
x = Valor(2.0); y = x ** 3; y.grad = 1.0; y._backward()
assert x.grad == 12.0, "d(x³)/dx = 3x² = 12 en x = 2"
x = Valor(0.5); y = x.tanh(); y.grad = 2.0; y._backward()
assert abs(x.grad - 2 * (1 - math.tanh(0.5) ** 2)) < 1e-12, "tanh' = 1 − tanh², multiplicado por sal.grad"
x = Valor(1.0); y = x.exp(); y.grad = 1.0; y._backward()
assert abs(x.grad - math.e) < 1e-12, "exp' = exp"
print("Correcto")

## Parte 3 · `backward` con orden topológico

Un nodo solo puede repartir su gradiente cuando ha recibido **todas** sus contribuciones. Por eso se recorre el grafo en orden topológico inverso: de la salida hacia las hojas.

**Ejercicio 3.** Construye `orden` con una búsqueda en profundidad (cada nodo se añade *después* de sus hijos), pon `self.grad = 1` y llama a `_backward()` de cada nodo en orden inverso.

In [ ]:
def backward(self):
    orden, visitados = [], set()
    # TU CÓDIGO
    pass

Valor.backward = backward

In [ ]:
# Comprobación con el ejemplo resuelto 2 del documento (sección 10.3): L = (a·b + c)·f
a, b, c, f = Valor(3.0), Valor(-2.0), Valor(8.0), Valor(-1.0)
L = (a * b + c) * f
L.backward()
assert L.dato == -2.0, "L debe valer −2"
assert [a.grad, b.grad, c.grad, f.grad] == [2.0, -3.0, -1.0, 2.0], "Esperaba ∂L/∂(a, b, c, f) = (2, −3, −1, 2)"
x = Valor(3.0); y = x * x + x; y.backward()
assert x.grad == 7.0, "x aparece tres veces: dy/dx = 2x + 1 = 7. ¿Usas += ?"
print("Correcto:", a, b, c, f)

## Parte 4 · Verificación contra PyTorch

PyTorch graba el mismo grafo con tensores. Usamos `float64` para comparar a precisión de máquina.

In [ ]:
x, y = Valor(0.5), Valor(-1.5)
z = (x * y + x.exp() / y ** 2).tanh() - x * 3
z.backward()

xt = torch.tensor(0.5, dtype=torch.float64, requires_grad=True)
yt = torch.tensor(-1.5, dtype=torch.float64, requires_grad=True)
zt = torch.tanh(xt * yt + torch.exp(xt) / yt ** 2) - xt * 3
zt.backward()
for nom, mio, pt in [("z", z.dato, zt.item()), ("dz/dx", x.grad, xt.grad.item()), ("dz/dy", y.grad, yt.grad.item())]:
    print(f"{nom:6s} Valor {mio:.10f}   PyTorch {pt:.10f}")
    assert abs(mio - pt) < 1e-12

**Ejercicio 4.** Con `Valor`, construye la entropía cruzada de los logits $(2, 1, 0)$ con clase correcta 0, $\mathcal L = -\log\big(e^{z_0}/\sum_j e^{z_j}\big)$. Llama a `backward` y guarda los tres gradientes en `grads`.

In [ ]:
z = [Valor(2.0), Valor(1.0), Valor(0.0)]
# TU CÓDIGO
L = None
grads = None

In [ ]:
# Comprobación
zt = torch.tensor([[2.0, 1.0, 0.0]], requires_grad=True)
Lt = F.cross_entropy(zt, torch.tensor([0])); Lt.backward()
assert L is not None and abs(L.dato - Lt.item()) < 1e-6, "La pérdida debe valer 0,4076"
assert np.allclose(grads, zt.grad[0].numpy(), atol=1e-6), "El gradiente debe ser softmax(z) − one_hot(0)"
print("Correcto. Pérdida", round(L.dato, 4), "· gradientes", [round(g_, 4) for g_ in grads])

## Parte 5 · Una red hecha con `Valor`

Con estas piezas ya se entrena una red. Cada neurona calcula $\tanh(\sum_i w_i x_i + b)$.

In [ ]:
class Neurona:
    def __init__(self, n_ent):
        self.w = [Valor(random.uniform(-1, 1)) for _ in range(n_ent)]
        self.b = Valor(0.0)
    def __call__(self, x):
        return sum((wi * xi for wi, xi in zip(self.w, x)), self.b).tanh()
    def parametros(self):
        return self.w + [self.b]

class Capa:
    def __init__(self, n_ent, n_sal):
        self.neuronas = [Neurona(n_ent) for _ in range(n_sal)]
    def __call__(self, x):
        return [n(x) for n in self.neuronas]
    def parametros(self):
        return [p for n in self.neuronas for p in n.parametros()]

class MLP:
    def __init__(self, n_ent, tamanos):
        dims = [n_ent] + tamanos
        self.capas = [Capa(dims[i], dims[i + 1]) for i in range(len(tamanos))]
    def __call__(self, x):
        for c in self.capas:
            x = c(x)
        return x[0] if len(x) == 1 else x
    def parametros(self):
        return [p for c in self.capas for p in c.parametros()]

random.seed(1)
red = MLP(3, [4, 4, 1])
xs = [[2.0, 3.0, -1.0], [3.0, -1.0, 0.5], [0.5, 1.0, 1.0], [1.0, 1.0, -1.0]]
ys = [1.0, -1.0, -1.0, 1.0]
print(len(red.parametros()), "parámetros")   # 3·4+4 + 4·4+4 + 4+1 = 41

**Ejercicio 5.** Haz 100 pasos de descenso de gradiente (η = 0,05) con pérdida $\sum_k (\hat y_k - y_k)^2$. En cada paso: hacia delante, **gradientes a cero**, `backward`, actualizar `p.dato`. Guarda la última pérdida en `perdida_final`.

In [ ]:
# TU CÓDIGO
perdida_final = None

In [ ]:
# Comprobación
assert perdida_final is not None and perdida_final < 0.05, "Tras 100 pasos la pérdida debería bajar de ≈ 4,5 a < 0,05. ¿Pones grad a cero?"
print("Correcto. Pérdida final:", round(perdida_final, 4), "· predicciones:", [round(red(x).dato, 3) for x in xs])

## Parte 6 · Forward frente a reverse

Tomamos $f(x_1,\dots,x_n) = \sum_i \tanh(x_i\,x_{i+1})$. Reverse da las $n$ derivadas con **una** pasada atrás; forward necesita $n$ pasadas con duales, cada una de coste ∝ n.

In [ ]:
def f_cadena(xs):
    return sum((xs[i] * xs[i + 1]).tanh() for i in range(len(xs) - 1))

for n in [25, 50, 100, 200]:
    x0 = [random.uniform(-1, 1) for _ in range(n)]
    t = time.perf_counter()
    vs = [Valor(v) for v in x0]; f_cadena(vs).backward()
    g_rev = [v.grad for v in vs]; t_rev = time.perf_counter() - t
    t = time.perf_counter()
    g_fwd = [f_cadena([Dual(v, float(j == i)) for j, v in enumerate(x0)]).b for i in range(n)]
    t_fwd = time.perf_counter() - t
    assert np.allclose(g_rev, g_fwd)
    print(f"n = {n:3d}   reverse {1000*t_rev:6.2f} ms   forward {1000*t_fwd:7.2f} ms   cociente {t_fwd/t_rev:5.1f}")

**Pregunta.** ¿Cómo crece el cociente con $n$? Un LLM tiene entre $10^9$ y $10^{12}$ parámetros: ¿qué supondría entrenarlo en modo forward?

## Parte 7 · PyTorch en cuatro ideas

1. **Tensor**: un array como los de NumPy, con dispositivo (CPU/GPU) y, si quieres, `requires_grad`.
2. **Autograd**: cada operación graba un nodo (`grad_fn`); `backward()` recorre el grafo como tu `Valor`.
3. **`nn.Module`**: agrupa parámetros y define `forward`.
4. **Optimizador**: `zero_grad()` y `step()`.

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
b = (a ** 2).sum()
print(b, "·", b.grad_fn, "←", b.grad_fn.next_functions[0][0])
b.backward(); print("db/da =", a.grad)               # 2a
(a ** 2).sum().backward(); print("Sin zero_grad se acumula:", a.grad)
with torch.no_grad():                                # no graba el grafo: evaluar o actualizar pesos
    print("requires_grad dentro de no_grad:", (a * 2).requires_grad)
print([(n, tuple(p.shape)) for n, p in nn.Linear(3, 2).named_parameters()])

## Parte 8 · El MLP de caracteres en PyTorch

Mismos datos y nombres que en los módulos 1 y 9: `nombres`, `vocab`, `stoi`, `itos`, partición con semilla 7, `B = 3`, d = 10, H = 64. En el módulo 9 escribiste a mano `C`, `W1`, `b1`, `W2`, `b2` y sus derivadas. Ahora `nn.Embedding` hace de `C`, `nn.Linear` de cada par (W, b) y autograd de la retropropagación.

In [ ]:
nombres = """
alejandro antonio manuel jose francisco david juan javier daniel carlos jesus miguel rafael pablo
pedro angel sergio fernando jorge luis alberto alvaro adrian diego raul enrique ramon ivan ruben
oscar andres vicente joaquin santiago victor eduardo mario roberto jaime ignacio marcos alfonso
jordi hugo ricardo salvador guillermo gabriel emilio gonzalo julio julian martin tomas marc agustin
nicolas samuel felix ismael cristian aitor joan josep lucas mateo leo bruno enzo izan alex gael
thiago marco dario unai iker eric rodrigo aaron nil pau oliver liam adam biel mauro saul cesar
lorenzo benito esteban felipe gregorio hector horacio jacinto leandro lisandro matias nestor octavio
patricio ramiro sebastian teodoro ulises valentin xavier yago zacarias aurelio baltasar ceferino
domingo elias fabian gaspar hilario iñigo nuño toño maria carmen ana isabel laura cristina marta
lucia paula elena sara raquel rosa pilar manuela concepcion mercedes julia beatriz nuria silvia
irene patricia rocio andrea montserrat monica alba sonia sandra marina angela sofia alicia susana
yolanda claudia natalia margarita eva inmaculada noelia veronica esther carla nerea daniela
alejandra martina valeria chloe emma olivia jimena vega lola carlota mia valentina aitana noa alma
triana candela ainhoa ines adriana aroa blanca celia diana elsa fatima gloria helena iria lara lidia
lorena luna maite mar miriam nadia nieves noemi olga paloma regina rebeca salma tamara teresa ursula
victoria ximena zoe aurora begoña catalina dolores encarnacion francisca gema juana leire macarena
nayara oihane piedad remedios soledad trinidad vanesa yaiza amparo araceli asuncion cecilia consuelo
emilia esperanza estefania eugenia guadalupe josefa lourdes maribel purificacion rosario virginia
almudena ariadna berta carolina debora estela fernanda graciela jazmin magdalena ofelia pepa rut
sabina tatiana yasmina zaira ainara arantxa iratxe naia uxue ane june maialen itziar alonso ernesto
marcelo bernardo cayetano rosalia anabel candido clemente efren genaro isidro leonardo maximo
norberto rogelio silvestre tadeo amelia bianca cintia dafne erika flor ivonne judith karen leticia
micaela nora perla rita selena telma vera wendy abril
""".split()

vocab = list(".abcdefghijklmnñopqrstuvwxyz")
V = len(vocab)
stoi = {c: i for i, c in enumerate(vocab)}
itos = {i: c for i, c in enumerate(vocab)}
rng = np.random.default_rng(7)
idx = rng.permutation(len(nombres))
n_val = round(0.15 * len(nombres))
val = [nombres[i] for i in idx[:n_val]]
train = [nombres[i] for i in idx[n_val:]]
print(len(nombres), "nombres ·", len(train), "entrenamiento ·", len(val), "validación")

In [ ]:
B = 3   # longitud del contexto

def construir_dataset(palabras, B=3):
    X, Y = [], []
    for p in palabras:
        ctx = [0] * B
        for c in p + ".":
            X.append(ctx); Y.append(stoi[c]); ctx = ctx[1:] + [stoi[c]]
    return torch.tensor(X).view(-1, B), torch.tensor(Y)

Xtr, Ytr = construir_dataset(train, B)
Xva, Yva = construir_dataset(val, B)

# Línea base del módulo 9: bigrama con suavizado α = 0,2 (el mejor en validación)
X1, Y1 = construir_dataset(train, 1)
N = torch.zeros(V, V).index_put_((X1[:, 0], Y1), torch.ones(len(Y1)), accumulate=True)
P = (N + 0.2) / (N + 0.2).sum(1, keepdim=True)
X1v, Y1v = construir_dataset(val, 1)
PPL_BIGRAMA = math.exp(-torch.log(P[X1v[:, 0], Y1v]).mean().item())
print(Xtr.shape, Xva.shape, f"· PPL validación del bigrama = {PPL_BIGRAMA:.2f}")

**Ejercicio 6.** Completa `MLPCaracteres`: `self.emb = nn.Embedding(V, d)`, concatenar los B vectores con `view`, `self.oculta = nn.Linear(B·d, H)` con ReLU y `self.salida = nn.Linear(H, V)`, que devuelve logits.

In [ ]:
class MLPCaracteres(nn.Module):
    def __init__(self, V, d=10, B=3, H=64):
        super().__init__()
        # TU CÓDIGO
    def forward(self, X):
        return None  # TU CÓDIGO

In [ ]:
# Comprobación
torch.manual_seed(0)
modelo = MLPCaracteres(V)
n_par = sum(p.numel() for p in modelo.parameters())
assert n_par == 4084, f"Esperaba 28·10 + 30·64 + 64 + 64·28 + 28 = 4.084 parámetros (como en el módulo 9); hay {n_par}"
assert modelo(Xtr[:5]).shape == (5, V), "La salida debe tener forma (lote, V)"
print("Correcto:", n_par, "parámetros")

**Ejercicio 7.** Entrena 1.000 pasos con lote completo, `torch.optim.SGD(lr=0.5, weight_decay=5e-3)` (el mismo η y λ del módulo 9) y `F.cross_entropy`. Guarda cada pérdida en `historial` y calcula `ppl_val` dentro de `torch.no_grad()`.

In [ ]:
opt = torch.optim.SGD(modelo.parameters(), lr=0.5, weight_decay=5e-3)
historial = []
# TU CÓDIGO
ppl_val = None

In [ ]:
# Comprobación
assert len(historial) == 1000 and historial[-1] < historial[0], "La pérdida debe bajar durante el entrenamiento"
assert ppl_val is not None and ppl_val < PPL_BIGRAMA, "El MLP debe superar al bigrama en validación"
print(f"Correcto. PPL validación: bigrama {PPL_BIGRAMA:.2f} · MLP {ppl_val:.2f}")
plt.plot(np.exp(historial)); plt.axhline(PPL_BIGRAMA, ls="--", c="gray", label="bigrama (validación)")
plt.yscale("log"); plt.xlabel("paso"); plt.ylabel("PPL de entrenamiento"); plt.legend(); plt.show()

In [ ]:
# Generar nombres con el modelo de PyTorch
g = torch.Generator().manual_seed(42)
muestras = []
for _ in range(12):
    ctx, out = [0] * B, ""
    while len(out) < 20:
        with torch.no_grad():
            p = F.softmax(modelo(torch.tensor([ctx])), dim=1)
        j = torch.multinomial(p, 1, generator=g).item()
        if j == 0:
            break
        out += itos[j]; ctx = ctx[1:] + [j]
    muestras.append(out)
print(muestras)

## Parte 9 · Retos

1. **ReLU y sigmoide en `Valor`.** Añádelas con su `_backward` y verifícalas contra PyTorch.
2. **El grafo.** Recorre `_prev` y `_op` para listar los nodos de `z` en la parte 4. ¿Cuántos hay? ¿Cuántos son hojas?
3. **AdamW.** Cambia el optimizador por `torch.optim.AdamW(lr=1e-2, weight_decay=0.1)` durante 400 pasos y mira la PPL de validación cada 100. ¿Qué pasa? Relaciónalo con el sobreajuste (módulo 8).
4. **Minibatches.** Entrena con lotes de 32 ejemplos (`torch.randint`) y compara (módulo 7).
5. **Inicialización del módulo 9.** Inicializa `oculta.weight` con He y `salida.weight` con desviación 0,1/√H usando `nn.init`. ¿Cambia la pérdida inicial? (Ojo: `nn.Linear` guarda W transpuesta, de forma (salida, entrada).)

---
### Soluciones de referencia
Inténtalo antes de mirar.

In [ ]:
# ---- Ejercicio 1
# __mul__:  return Dual(self.a * o.a, self.a * o.b + self.b * o.a)
# tanh:     t = math.tanh(self.a);  return Dual(t, (1 - t * t) * self.b)
# ---- Ejercicio 2
# mul:   self.grad += otro.dato * sal.grad;  otro.grad += self.dato * sal.grad
# pow:   self.grad += k * self.dato ** (k - 1) * sal.grad
# tanh:  self.grad += (1 - t ** 2) * sal.grad
# exp:   self.grad += e * sal.grad
# ---- Ejercicio 3
#     def construir(v):
#         if v not in visitados:
#             visitados.add(v)
#             for h in v._prev:
#                 construir(h)
#             orden.append(v)
#     construir(self)
#     self.grad = 1.0
#     for v in reversed(orden):
#         v._backward()
# ---- Ejercicio 4
# e = [zi.exp() for zi in z];  L = -(e[0] / sum(e)).log();  L.backward()
# grads = [zi.grad for zi in z]
# ---- Ejercicio 5
# for paso in range(100):
#     L = sum((red(x) - y) ** 2 for x, y in zip(xs, ys))
#     for p in red.parametros(): p.grad = 0.0
#     L.backward()
#     for p in red.parametros(): p.dato -= 0.05 * p.grad
# perdida_final = L.dato
# ---- Ejercicio 6
#     __init__: self.emb = nn.Embedding(V, d); self.oculta = nn.Linear(B * d, H); self.salida = nn.Linear(H, V)
#     forward:  E = self.emb(X).view(X.shape[0], -1);  return self.salida(F.relu(self.oculta(E)))
# ---- Ejercicio 7
# for paso in range(1000):
#     loss = F.cross_entropy(modelo(Xtr), Ytr)
#     opt.zero_grad(); loss.backward(); opt.step()
#     historial.append(loss.item())
# with torch.no_grad():
#     ppl_val = math.exp(F.cross_entropy(modelo(Xva), Yva).item())